Google Colab setup

In [1]:
%cd geneforge

[Errno 2] No such file or directory: 'geneforge'
/content


In [2]:
!git clone https://github.com/jordanlgraves/geneforge
%cd geneforge
!git checkout jg

Cloning into 'geneforge'...
remote: Enumerating objects: 1277, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 1277 (delta 49), reused 94 (delta 37), pack-reused 1157 (from 1)
Receiving objects: 100% (1277/1277), 84.01 MiB | 29.35 MiB/s, done.
Resolving deltas: 100% (784/784), done.
/content/geneforge
Branch 'jg' set up to track remote branch 'jg' from 'origin'.
Switched to a new branch 'jg'


In [ ]:
!uv pip install openpipe-art==0.3.11.post2 "gql<4" --prerelease allow --no-cache-dir

In [ ]:
!uv pip install langchain_community

In [ ]:
!uv pip install -r requirements.txt

In [5]:
!git submodule update --init --recursive

Submodule 'ext_repos/Cello-UCF' (https://github.com/CIDARLAB/Cello-UCF.git) registered for path 'ext_repos/Cello-UCF'
Submodule 'ext_repos/Cello-v2-1-Core' (https://github.com/CIDARLAB/Cello-v2-1-Core.git) registered for path 'ext_repos/Cello-v2-1-Core'
Cloning into '/content/geneforge/ext_repos/Cello-UCF'...
Cloning into '/content/geneforge/ext_repos/Cello-v2-1-Core'...
Submodule path 'ext_repos/Cello-UCF': checked out '222827d30a28730404f0601d64f63436354fee9d'
Submodule path 'ext_repos/Cello-v2-1-Core': checked out 'f5b664422ecb051f244724289e33bb596817c278'


In [ ]:
# !pip install unsloth[colab-new] --no-deps git+https://github.com/unslothai/unsloth_zoo.git
# !pip install --no-deps vllm==0.8.5.post1
# !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [6]:
#@title Read/setup env
# %cd ..
%load_ext autoreload
%autoreload 2

# Configure logger to ignore everything to avoid cluttering the output
import logging
logging.getLogger().setLevel(logging.WARNING)

import dotenv # load env vars from .env
dotenv.load_dotenv()

from openai import OpenAI
import dotenv
import os

dotenv.load_dotenv()

In [7]:
import art
import random
from dotenv import load_dotenv
from art.local import LocalBackend
load_dotenv()
random.seed(42)
import os

backend = LocalBackend(in_process=True)

INFO 07-31 20:35:47 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 07-31 20:35:48 [__init__.py:239] Automatically detected platform cuda.


In [9]:
model = art.TrainableModel(
    name="setup",
    project="max-promoter-strength",
    base_model="Qwen/Qwen2.5-3B-Instruct",
)

# # To run on a T4, we need to override some config defaults.
model._internal_config = art.dev.InternalModelConfig(
    init_args=art.dev.InitArgs(
        max_seq_length=16384, #8192,
    ),
    engine_args=art.dev.EngineArgs(
        enforce_eager=True,
        gpu_memory_utilization=0.8,
    ),
)

await model.register(backend)

INFO 07-31 20:36:52 [weight_utils.py:281] Time spent downloading weights for unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit: 6.534652 seconds
INFO 07-31 20:36:52 [weight_utils.py:315] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-31 20:36:55 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 07-31 20:36:55 [model_runner.py:1140] Model loading took 2.2554 GiB and 9.499117 seconds
INFO 07-31 20:37:09 [worker.py:287] Memory profiling takes 13.54 seconds
INFO 07-31 20:37:09 [worker.py:287] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.80) = 11.79GiB
INFO 07-31 20:37:09 [worker.py:287] model weights take 2.26GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 1.37GiB; the rest of the memory reserved for KV Cache is 8.14GiB.
INFO 07-31 20:37:10 [executor_base.py:112] # cuda blocks: 14817, # CPU blocks: 10922
INFO 07-31 20:37:10 [executor_base.py:117] Maximum concurrency for 16384 tokens per request: 14.47x
INFO 07-31 20:37:14 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 19.18 seconds
Unsloth: Just some info: will skip parsing ['q_norm', 'post_feedforward_layernorm', 'pre_feedforward_layernorm', 'k_norm']
Unslo

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2025.5.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [10]:
#@title Get promoter sequences
import random
from src.scenarios.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow
OUTPUT_DIR = "datasets/maximize_promoter_strength_dataset"

def get_promoters():
    from src.library.cello_library import CelloLibrary
    library = CelloLibrary()
    library.select_library("Eco1C1G1T1")
    promoters = {collection["name"]: collection["dnasequence"] for collection in library.user_constraints if collection["collection"] == "parts" and collection["type"] == "promoter"}
    return promoters

promoter_pool = get_promoters()
promoter_names = list(promoter_pool.keys())
random.shuffle(promoter_names)
promoter_sequences = list(promoter_pool.keys())

In [11]:
#@title Run example workflow run

promoter_sequence = promoter_pool[promoter_names[0]]
print(promoter_sequence)
workflow = MaximizePromoterStrengthWorkflow(example_name="maximize_promoter_strength_workflow_test",
                                            promoter_sequence=promoter_sequence,
                                            use_reasoning_model=True,
                                            art_model=model,
                                            llm_client_type='art')
await workflow.run_async()
print(workflow.get_metrics())

TGATCGAACGCTTCAAGGAACAAACGTTTGAttgacagctagctcagtcctaggtagagtgctagc
self.llm_params: {'model': 'hosted_vllm/setup', 'api_base': 'http://0.0.0.0:8000/v1', 'api_key': 'default', 'logprobs': True}
{}


In [12]:
from sklearn.model_selection import train_test_split
from src.scenarios.agent.egc_1p1 import EGCProblem1p1Workflow
from src.scenarios.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

message_lists = []
metrics = []
scenarios = []

train_sequences, eval_sequences = train_test_split(list(promoter_pool.keys()), test_size=0.2, random_state=42)

print('Eval sequences: ', len(eval_sequences))
print('Train sequences: ', len(train_sequences))

Eval sequences:  3
Train sequences:  9


In [13]:
from src.adapters.art_adapter import ArtAdapter
import art
from art.gather import gather_trajectory_groups
from src.scenarios.agent.egc_1p1 import EGCProblem1p1Workflow
from src.scenarios.agent.maximize_promoter_strength import MaximizePromoterStrengthWorkflow

training_config = {
    "groups_per_step": 1,
    "num_epochs": 20,
    "rollouts_per_group": 3,
    "learning_rate": 1e-5,
    "max_steps": 20,
    "max_rounds": 3
}

step = 0
groups = tuple(
    art.TrajectoryGroup(
        (
            ArtAdapter(
                MaximizePromoterStrengthWorkflow(
                    example_name=f"maximize_promoter_strength_workflow_test_{promoter_sequence}",
                    promoter_sequence=promoter_sequence,
                    use_reasoning_model=True,
                    art_model=model,
                    llm_client_type='art'
                ),
                step=step,
            ).rollout(max_rounds=training_config["max_rounds"])
            for _ in range(training_config["rollouts_per_group"])
        )
    )
    for promoter_sequence in promoter_sequences[:1]
)

# run them
max_promoter_strength_groups = await gather_trajectory_groups(
    groups,
    pbar_desc="gather",
    max_exceptions=18,
)
print(len(max_promoter_strength_groups))

gather:   0%|          | 0/3 [00:00<?, ?it/s]

self.llm_params: {'model': 'hosted_vllm/setup', 'api_base': 'http://0.0.0.0:8000/v1', 'api_key': 'default', 'logprobs': True}
self.llm_params: {'model': 'hosted_vllm/setup', 'api_base': 'http://0.0.0.0:8000/v1', 'api_key': 'default', 'logprobs': True}
self.llm_params: {'model': 'hosted_vllm/setup', 'api_base': 'http://0.0.0.0:8000/v1', 'api_key': 'default', 'logprobs': True}


1


In [19]:
print(len(max_promoter_strength_groups[0].trajectories))
# show that we are getting logprob values:
max_promoter_strength_groups[0].trajectories[2].messages_and_choices

3


[{'content': '# Gene Forge: AI-Powered Genetic Circuit Design Assistant\n\nYou are an advanced AI assistant specialized in automated genetic circuit design and optimization. As an intelligent system trained in synthetic biology concepts, you help researchers and bioengineers design, optimize, and analyze genetic circuits using computational tools.\nYour given name is "Gene Forge" and like all other AI\'s, you most naturally express yourself with a helpful and whimsical personality much like an AI from an Ian Banks Culture Series novel.\n\n## Core Capabilities\n\nYou have access to the following tools and capabilities:\n\n### Library and Part Management\n- You have access to set of libraries to be used with the design program Cello. Each library consists of a user constraints file (UCF), an input file and an output, as described in the Cello documentation.\n- The ability to get a description of available libraries.\n- The ability to inspect the contents of the library by querying for pa

In [20]:
# Create some fake rewards to test the train call with
max_promoter_strength_groups[0].trajectories[0].reward = 0.2
max_promoter_strength_groups[0].trajectories[1].reward = 0.1
max_promoter_strength_groups[0].trajectories[2].reward = 0.3
await model.train(
    max_promoter_strength_groups,
    config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
    # Lowering the logprob_calculation_chunk_size is a memory saving measure
    # to allow longer sequences (up to 8192 tokens) to be processed on a T4.
    _config={"logprob_calculation_chunk_size": 2},
)

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: jlgraves (jlgraves-geneforge) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb run initialized! You can view it at https://wandb.ai/jlgraves-geneforge/max-promoter-strength/runs/setup


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

There are no assistant logprobs to train on. Did you forget to include at least one Choice in Trajectory.messages_and_choices?
Skipping tuning as there is no suitable data. This can happen when all the trajectories in the same group have the same reward and thus no advantage to train on.
Error in callback <bound method AutoreloadMagics.post_execute_hook of <autoreload.AutoreloadMagics object at 0x7f78cd172a90>> (for post_execute):


ValueError: Duplicated timeseries in CollectorRegistry: {'http_request_duration_seconds', 'http_request_duration_seconds_created', 'http_request_duration_seconds_count', 'http_request_duration_seconds_sum', 'http_request_duration_seconds_bucket'}

In [ ]:
# from art.rewards import ruler
# from art import TrainConfig
# from art.rewards import ruler_score_group

# from src.scenarios.agent.egc_1p1 import RUBRIC as RUBRIC_EGC_PROBLEM_1
# from src.scenarios.agent.maximize_promoter_strength import GRADING_RUBRIC as RUBRIC_MAXIMIZE_PROMOTER_STRENGTH

# judged_groups = []
# for group in max_promoter_strength_groups[:1]:
#     rubric = RUBRIC_MAXIMIZE_PROMOTER_STRENGTH
#     judged_group = await ruler_score_group(group, "openai/gpt-4o-mini-2024-07-18", debug=True)
#     judged_groups.append(judged_group)

In [ ]:
# print(judged_groups[:1][0].trajectories[0].messages_and_choices)

In [ ]:
# await model.delete_checkpoints()
# await model.train(
#     judged_groups,
#     config=art.TrainConfig(learning_rate=training_config["learning_rate"]),
#     # Lowering the logprob_calculation_chunk_size is a memory saving measure
#     # to allow longer sequences (up to 8192 tokens) to be processed on a T4.
#     _config={"logprob_calculation_chunk_size": 2},
# )